# FedXCrop: running the experiment grid

Runs the full experiment grid on a GPU runtime and writes every result back to Google Drive, so a disconnected session resumes instead of restarting.

Set the runtime to a GPU before running: Runtime, Change runtime type, T4 GPU or better.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, change the runtime type')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive

Checkpoints and results live on Drive so nothing is lost when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/fedxcrop'
!mkdir -p {WORK}

## 2. Get the code

The repository holds the code, the fixed splits, and the client partitions. It does not hold the dataset.

In [ ]:
%cd /content
!git clone https://github.com/Mayan10/crp.git fedxcrop_repo || (cd fedxcrop_repo && git pull)
%cd /content/fedxcrop_repo
!git log --oneline -1

In [ ]:
!pip install -q captum imagehash 'flwr[simulation]==1.11.1'

import torch, torchvision, flwr
print('torch', torch.__version__, 'torchvision', torchvision.__version__)
print('flwr', flwr.__version__)

# Flower's simulation engine needs Ray. If this import fails the grid cannot
# run on the Flower engine, and every run below needs --engine sequential.
import ray
print('ray', ray.__version__)


## 3. Get the dataset

Either copy it from Drive (fastest if you already uploaded it) or download it from Kaggle. The split CSVs reference paths of the form `color/<class>/<file>.JPG`, so the directory that holds `color/`, `segmented/` and `grayscale/` is the dataset root.

In [ ]:
import os

DATASET = '/content/plantvillage dataset'
DRIVE_ARCHIVE = f'{WORK}/plantvillage.zip'

if not os.path.isdir(f'{DATASET}/color'):
    if os.path.exists(DRIVE_ARCHIVE):
        !unzip -q '{DRIVE_ARCHIVE}' -d /content/
    else:
        # Needs ~/.kaggle/kaggle.json, see the Kaggle account page for the token.
        !pip install -q kaggle
        !kaggle datasets download -d amgadalameri/plantvillage-dataset-zip -p /content --unzip

print('color classes:', len(os.listdir(f'{DATASET}/color')))

In [ ]:
# Verify the dataset matches the committed splits before spending GPU hours on it.
!python -c "\
import pandas as pd, pathlib; \
root = pathlib.Path('$DATASET'); \
t = pd.read_csv('splits/test.csv'); \
missing = [p for p in t['path'].head(500) if not (root / p).is_file()]; \
print('missing of first 500 test images:', len(missing)); \
print('dataset root looks correct' if not missing else 'PATHS DO NOT MATCH, check the root')"

## 4. Point runs and results at Drive

`runs/` holds checkpoints and is where a resumed run picks up. `results/` holds the small files that get committed back.

In [ ]:
!mkdir -p {WORK}/runs {WORK}/results
!rm -rf /content/fedxcrop_repo/runs && ln -s {WORK}/runs /content/fedxcrop_repo/runs
!ls -la /content/fedxcrop_repo/runs/ | head -3

# federated.engine defaults to flower in configs/base.yaml. Ray gives each
# client the whole GPU in turn, so clients run one at a time on one device.
OVERRIDES = f"data.root='{DATASET}' data.num_workers=2 device=cuda"
print(OVERRIDES)


## 5. Smoke test

Two minutes, and it fails loudly if anything above is wrong. Do not skip it.

In [ ]:
# Both engines, then the test that they implement one method rather than two.
!python scripts/run_federated.py --config configs/fedprox_noniid.yaml --smoke \
  --no-resume --engine sequential --set data.root="$DATASET"
!python scripts/run_federated.py --config configs/fedprox_noniid.yaml --smoke \
  --no-resume --engine flower --set data.root="$DATASET"


In [ ]:
# Run this before spending GPU hours. It checks the Flower engine against the
# sequential reference: same shard sizes, same aggregation weighting, same
# communication accounting, and accuracy landing in the same place.
!python -m pytest tests/ -q -m "not slow"
!python -m pytest tests/test_smoke.py -q -m slow -k "flower or agree"


If `test_both_engines_agree` fails, stop and fix it before running the grid:
it means the two engines are not running the same method, and the grid would
produce numbers that cannot be reconciled with the unit tested reference.


## 6. Time one round

Measure before committing to the whole grid, so the total is known rather than guessed.

In [ ]:
import time
start = time.time()
!python -u scripts/run_federated.py --config configs/fedavg_iid.yaml --no-resume \
  --set federated.rounds=1 {OVERRIDES}
per_round = time.time() - start
print(f'\none round including startup: {per_round / 60:.1f} min')
print(f'estimated core grid (24 federated runs x 30 rounds): {24 * 30 * per_round / 3600:.1f} h')
print(f'estimated centralized (3 seeds x 20 epochs):          {3 * 20 * per_round / 3600:.1f} h')

## 7. Run the grid

Restartable: finished runs are skipped and an interrupted run resumes from its last completed round. If the session drops, rerun this cell after remounting.

In [ ]:
!python -u scripts/run_grid.py --group centralized --set {OVERRIDES}

In [ ]:
!python -u scripts/run_grid.py --group fedavg_iid --set {OVERRIDES}
!python -u scripts/run_grid.py --group fedavg_noniid --set {OVERRIDES}

In [ ]:
# Selects mu on validation accuracy at alpha 0.1, seed 0 only.
!python -u scripts/run_grid.py --group mu_selection --set {OVERRIDES}
!python scripts/select_mu.py

In [ ]:
# Use the mu that scripts/select_mu.py reported.
SELECTED_MU = 0.01
!python -u scripts/run_grid.py --group fedprox_noniid --mu {SELECTED_MU} --set {OVERRIDES}

In [ ]:
# Reproduces the original protocol for comparison. Needs the centralized seed 0 run.
!python -u scripts/run_grid.py --group legacy --set {OVERRIDES}

## 8. Explainability, figures, and tables

In [ ]:
!python -u scripts/run_xai.py --models centralized_seed0 \
  fedavg_dirichlet_alpha0.1_K5_seed0 fedprox_dirichlet_alpha0.1_K5_mu{SELECTED_MU}_seed0 \
  fedavg_dirichlet_alpha0.5_K5_seed0 fedprox_dirichlet_alpha0.5_K5_mu{SELECTED_MU}_seed0 \
  --sanity --set {OVERRIDES}

In [ ]:
!python scripts/make_figures.py --set {OVERRIDES}
!python scripts/check_pseudo_masks.py --set data.root="$DATASET"

# Attribution grids (figures 4 and 5) and the failure cases (figure 7).
!python scripts/make_xai_figures.py --set {OVERRIDES} --models \
  centralized_seed0 fedprox_dirichlet_alpha0.1_K5_mu{SELECTED_MU}_seed0


## 9. Save the results back

Copies the small result files to Drive. Commit them from your own machine so the commits carry your identity.

In [ ]:
!rsync -a --exclude '*.pt' /content/fedxcrop_repo/results/ {WORK}/results/
!du -sh {WORK}/results
print('Download or sync this folder, copy it into the repo results/, then commit locally.')